In [1]:
import pandas as pd
from pathlib import Path

# 🗺️ Mapping from PROJECT SITE to REGION LWDB
region_lwdb_map = {
   "Golden Sierra Workforce Development Boards": "Golden Sierra Workforce Development Board",
   "Los Angeles City Workforce Development Board": "City of Los Angeles Workforce Development Board",
   "Madera County Workforce Development Board": "Workforce Development Board of Madera County",
   "North Central Counties Consortium (NCCC)": "North Central Counties Consortium",
   "Sacramento Employment and Training Agency": "Sacramento Employment And Training Agency",
   "San Bemardino County Workforce Development Board": "San Bernardino County Workforce Development Department",
   "Solano County Workforce Development Board": "Workforce Investment Board of Solano County",
   "Ventura County Workforce Development Board": "Workforce Development Board of Ventura County",
   "Southeast Los Angeles County Workforce Development Board": "Southeast Los Angeles Workforce Development Board", 
   "Humboldt County Workforce Development Board": "Humboldt County Workforce Development Board",
   "South Bay Workforce Investment Board": "South Bay Workforce Investment Board", 
   "San Joaquin County Workforce Development Board": "San Joaquin County Workforce Development Board", 
   "Alameda County Workforce Development Board": "Alameda County Workforce Development Board",
   "Yolo County Health And Human Services Agency (Hhsa)": "Yolo County Workforce Development Board"
}
REGION_CODE_LOOKUP = {
   "Sacramento Employment And Training Agency":	29,
    "Workforce Investment Board of Solano County":	43,
    "Workforce Development Board of Ventura County":	48,
    "North Central Counties Consortium":	23,
    "Southeast Los Angeles Workforce Development Board":	42,
    "Workforce Development Board of Madera County":	15,
    "San Bernardino County Workforce Development Department":	32,
    "City of Los Angeles Workforce Development Board":	12,
    "Golden Sierra Workforce Development Board":	7, 
    "Yolo County Workforce Development Board":	50,
    "Humboldt County Workforce Development Board":	8,
    "South Bay Workforce Investment Board":	45,
    "San Joaquin County Workforce Development Board":	35,
    "Alameda County Workforce Development Board":	1, 
    "Mother Lode Workforce Development Board":110,
    "Northern Rural Training and Employment Consortium":120,
    "Riverside County Workforce Development Board":130,
    "San Diego Workforce Partnership":140
}
# User home directory (dynamic)
home_dir = Path.home()

# SharePoint relative path inside OneDrive
sharepoint_relative_input_path = Path(
    "APM US",
    "Data and Insights - Documents",
    "Power BI Source Files",
    "DOR-AJCC Collab",
    "Files for Python"
)
# 🔥 File for conversion
file_substring = "Enrollment Targets"

base_dir = home_dir / sharepoint_relative_input_path

# 🧐 Get files matching substring
matching_files = [
    f for f in base_dir.glob("*.xlsx")
    if file_substring in f.name
]
# ☠️ No files matching error message
if not matching_files:
    raise FileNotFoundError("No matching Excel file found.")
    
# ☠️ Muiltiple files matching substring error message
if len(matching_files) > 1:
    raise ValueError(
        f"Multiple matching files found:\n{[f.name for f in matching_files]}"
    )

input_path = matching_files[0]

print(f"✅ Using file: {input_path}")

# ✔️ Declare path to write new file to
# SharePoint relative path inside OneDrive
sharepoint_relative_output_path = Path(
    "APM US",
    "Data and Insights - Documents",
    "Power BI Source Files",
    "DOR-AJCC Collab"
)

output_base_dir = home_dir / sharepoint_relative_output_path

# ✔️ Full path with name for output file 
output_path = rf'{output_base_dir}\{file_substring}.xlsx'
# output_path = rf'C:\Users\StevenFoster\Downloads\{file_name}.xlsx'

# 🛻 Load the 1st worksheet with NO headers
df_raw = pd.read_excel(input_path, sheet_name=0, header=None)

# 🛻 Find the row index where column 0 == "#"
header_row_idx = df_raw.index[df_raw.iloc[:, 0] == "Enrollment Targets"]

# ☠️ Error message when column header not located
if header_row_idx.empty:
    raise ValueError("Header row containing 'Enrollment Targets' not found.")

header_row_idx = header_row_idx[0]

# ➖ Remove rows above the header
df_clean = df_raw.iloc[header_row_idx:].reset_index(drop=True)

# 🛻 Promote that row to headers
df_clean.columns = df_clean.iloc[0]
df_clean = df_clean.iloc[1:].reset_index(drop=True)

# ✔️ Clean column names
df_clean.columns = (
    df_clean.columns
        .astype(str)
        .str.replace('\n', ' ', regex=False)  # replace newline with space
        .str.replace(r'\s+', ' ', regex=True) # collapse multiple spaces
        .str.strip()                           # trim leading/trailing spaces
)

# ➖ Remove repeated header rows inside the data
df_clean = df_clean[df_clean["Enrollment Targets"] != "Enrollment Targets"]

# ➖ Remove total row
df_clean = df_clean[~df_clean["Enrollment Targets"].astype(str).str.contains(
    "Total", na=False
)]

# 🔄 Rename column header with # to Code
if "Enrollment Goals" in df_clean.columns:
    print("ℹ️ Renaming 'Enrollment Goals' → 'Enrollment Targets'")

    # Rename column
    df_clean = df_clean.rename(columns={"Enrollment Goals": "Enrollment Targets"})

# ➕ Define what columns should be kept the 1st worksheet with NO headers
cols_to_keep = [
    "Enrollment Targets",
    "SUB Code",
    "Site"
]

df = df_clean[cols_to_keep].copy()

# 4️⃣ Validate columns are present 
expected_string_cols = ["SUB Code", "Site"]
expected_numeric_cols = ["Enrollment Targets"]

for col in expected_string_cols:
    if col not in df.columns:
        raise ValueError(f"❌ Missing expected string column: '{col}'")

    # Convert to string
    df[col] = df[col].astype("string")

    # Verify all values are strings (or NA)
    bad_mask = df[col].apply(lambda v: not (pd.isna(v) or isinstance(v, str)))
    if bad_mask.any():
        bad_values = df.loc[bad_mask, col].head(10)
        raise TypeError(
            f"❌ Column '{col}' contains non-string data.\n"
            f"Example invalid values:\n{bad_values}"
        )
for col in expected_numeric_cols:
    if col not in df_clean.columns:
        raise ValueError(f"❌ Missing expected numeric column: '{col}'")

    # Try converting to numeric
    converted = pd.to_numeric(df_clean[col], errors="coerce")

    # If any value becomes NaN but original was not NaN → bad data
    bad_mask = converted.isna() & df_clean[col].notna()
    if bad_mask.any():
        bad_values = df_clean.loc[bad_mask, col].head(10)
        raise TypeError(
            f"❌ Column '{col}' contains non-numeric data.\n"
            f"Example invalid values:\n{bad_values}"
        )

    # Assign validated numeric version
    df_clean[col] = converted


print("✅ All column types validated successfully. Safe to continue.")

# ➕ Add REGION LWDB column with fallback to PROJECT SITE
df["REGION LWDB"] = df["Site"].map(region_lwdb_map).fillna(df["Site"])

df = df[df['Enrollment Targets'].notna()]

# # ➕ Normalize REGION LWDB for reliable matching
# df["REGION LWDB NORMALIZED"] = (
#     df["REGION LWDB"]
#         .str.upper()
#         .str.replace(r"[()]", "", regex=True)
#         .str.replace(r"\s+", " ", regex=True)
#         .str.strip()
# )

# ➕ Map REGION CODE
df["REGION CODE"] = df["REGION LWDB"].map(REGION_CODE_LOOKUP)

# unmapped = df.loc[df["REGION CODE"].isna(), "REGION LWDB"].unique()

# if len(unmapped) > 0:
#     raise ValueError(
#         f"❌ The following REGION LWDB values could not be mapped:\n{unmapped}"
#     )
    
# 🤖 Capitlize headers
df.columns = df.columns.str.upper()

# 🤖 Drop blank rows, fail back for "used range" in excel file
df = df.dropna(how="all")

# 🤖 Trim all string columns 
df = df.apply(
    lambda col: col.str.strip() if col.dtype == "string" else col
)

# 🤖 Convert output path to pathlib path 
output_path = Path(output_path)

# 🤖 Check if file exists and remove 
if output_path.exists():
    print(f"⚠️ Overwriting existing file: {output_path}")
    output_path.unlink()  # deletes the file
else: 
    print(f"🆕 File does not exist. Creating new file at: {output_path}")
    
# ✍️ Export to Excel
df.to_excel(output_path, index=False)

✅ Using file: C:\Users\StevenFoster\APM US\Data and Insights - Documents\Power BI Source Files\DOR-AJCC Collab\Files for Python\Enrollment Targets.xlsx
✅ All column types validated successfully. Safe to continue.
⚠️ Overwriting existing file: C:\Users\StevenFoster\APM US\Data and Insights - Documents\Power BI Source Files\DOR-AJCC Collab\Enrollment Targets.xlsx
